In [ ]:
import re
import json
import pandas as pd
import numpy as np
from datetime import datetime
import pycountry
from IPython.display import display, Markdown
from numpyro import distributions as dist
import yaml as yml
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
import git

from emu_renewal.constants import BASE_PATH, OUTPUTS_PATH
from emu_renewal.selection import get_mob_avail_countries, gather_who_data, find_absent_inds, \
    find_neg_inds, find_outliers, find_nans_repeats
from emu_renewal.inputs import get_worldbank_national_pop, get_income_group, get_fb_visited_mobility, \
    get_fb_singletile_mobility, process_raw_google_mobility, get_google_mobility, get_country_pop, \
    get_undesa_national_pop, get_country_vacc_data, get_world_shp
from emu_renewal.outputs import get_table_df_from_priors_dict, add_bool_row_to_table, add_mob_avail_to_world
from emu_renewal.document import get_func_blurb
from emu_renewal.indicators import get_deaths_target, get_cases_target, filter_seroprev, get_owid_hosps, \
    get_all_seroprev, get_seroprev_pooled_totals, get_var_target, extract_specific_var, get_country_vars, \
    get_alpha_info, get_delta_info, get_specific_var_props, get_ba2_target, get_ba5_target, \
    get_ba2_info, get_ba5_info
from emu_renewal.run import find_run_start_time, find_run_end_time, get_hosp_target, get_seroprev_target, \
    get_mobility_provider, run_calibration
from emu_renewal.geospatial import FacebookMobilityBuilder
from emu_renewal.renew import MultiStrainModel
from emu_renewal.distributions import GammaDens
from emu_renewal.process import _get_cos_curve_at_x
from emu_renewal.mobility import NoMobilityProvider, WeightedExpMobilityProvider, SingleSeriesExpMobilityProvider
from emu_renewal.calibration import custom_init, StandardCalib
from emu_renewal.priors import get_standard_priors
from emu_renewal.plotting import plot_inclusion

plt.style.use("default")
set_matplotlib_formats("svg")

In [ ]:
txt = "\n\n# Documentation\n"
commit = git.Repo(search_parent_directories=True).head
txt += (
    "The following document was documented algorithmically "
    "from the code used in the analysis.\n"
)
txt += f"This version was produced from commit {commit.object.hexsha}. \n\n"
txt += "# Analysis methods \n\n"
txt += "## Modelled population size \n\n"
txt += get_func_blurb(get_country_pop)
txt += get_func_blurb(get_worldbank_national_pop)
txt += get_func_blurb(get_undesa_national_pop)
txt += "\n\n## Analysis period\n"
analysis_period_rationale = (
    "For all countries, we aimed to place our analysis period "
    "during a time period for which the variation in transmission rate "
    "was likely to be predominantly attributable to changes in mobility, "
    "although we developed an approach (described below) to address "
    "the consideration that the emergence of new SARS-CoV-2 variants "
    "likely contributed substantially. "
    "Therefore, we aimed to select time periods during which "
    "the roll-out of vaccination programs would have contributed relatively "
    "less to variations in transmission rate.\n\n"
    "For all countries other than Australia, we addressed this "
    "by limiting our analysis period to the time period prior to "
    "vaccination reaching levels that may have had a significant "
    "population-level effect."
)
txt += analysis_period_rationale
txt += get_func_blurb(find_run_start_time) + "\n\n"
txt += get_func_blurb(find_run_end_time)
txt += get_func_blurb(get_country_vacc_data)
txt += "\n\n## Calibration targets\n"
indicator_rationale = (
    "We only included countries for which multiple "
    "data streams were available to estimate variations "
    "in transmission rates over time. "
    "Specifically, we required that a time series for both "
    "COVID-19-attributable deaths and case notifications were available. "
    "However, we also included one time series for a hospital-related "
    "indicator, where this could be sourced. "
    "Because we considered that the emergence of new "
    "SARS-CoV-2 variants could have substantially affect transmission, "
    "we included explicit calibration targets for the "
    "emergence of new variants where these could be sourced. "
    "Although it may have been less epidemiologically relevant to "
    "this analysis, we sought to broadly capture the "
    "overall size of the epidemic. This was addressed through "
    "the inclusion of seroprevalence estimates where available. "
    "We did not include vaccination explicitly within the model "
    "because our approach to country and time period inclusion "
    "was intended to focus on time periods during which "
    "vaccination would not have substantially influenced transmission rates.\n\n"
)
txt += indicator_rationale
txt += "\n\n### WHO indicators\n"
txt += get_func_blurb(get_deaths_target)
txt += get_func_blurb(get_cases_target)
txt += "\n\n### Hospitalisations\n"
txt += get_func_blurb(get_owid_hosps)
txt += get_func_blurb(get_hosp_target) + "\n\n"
txt += "\n\n### Seroprevalence\n"
txt += get_func_blurb(get_all_seroprev)
txt += get_func_blurb(filter_seroprev)
txt += get_func_blurb(get_seroprev_pooled_totals) + "\n\n"
txt += get_func_blurb(get_income_group)
txt += get_func_blurb(get_seroprev_target) + "\n\n"
txt += "\n\n### Variants\n\n"
variant_rationale = (
    "We included up to three key variants explicitly within "
    "our analysis framework. For all countries but Australia "
    "our approach was intended to represent strains prior to the "
    "emergence of the Alpha variant and strains of Alpha "
    "with strains of Delta also included where sufficient data "
    "were available and the time period simulated meant that "
    "the emergence of Delta was relevant to the simulation. "
    "For Australia, we included pre-BA.2 strains of Omicron, "
    "along with Omicron BA.2 and Omicron BA.5. \n\n"
)
txt += "#### Data extraction\n"
txt += get_func_blurb(get_country_vars)
txt += get_func_blurb(extract_specific_var)
txt += get_func_blurb(get_specific_var_props) + "\n\n"
txt += get_func_blurb(get_var_target)
txt += "\n\n#### Alpha\n"
txt += get_func_blurb(get_alpha_info)
txt += "\n\n#### Delta\n"
txt += get_func_blurb(get_delta_info)
txt += "\n\n#### Omicron BA.2\n"
txt += get_func_blurb(get_ba2_target)
txt += get_func_blurb(get_ba2_info)
txt += "\n\n#### Omicron BA.5\n"
txt += get_func_blurb(get_ba5_target)
txt += get_func_blurb(get_ba5_info)
txt += "\n\n## Mobility\n"
mobility_rationale = (
    "The main purpose of our analysis was to understand "
    "the effect of empirically observed changes in mobility "
    "on the observed fluctuations of the COVID-19 epidemic. "
    "We addressed this by including various mobility observation "
    "streams available from Big Tech companies within our model.\n\n"
)
txt += mobility_rationale
txt += get_func_blurb(get_mobility_provider)
txt += "\n\n### No mobility\n"
no_mob_provider = NoMobilityProvider()
txt += get_func_blurb(NoMobilityProvider.__init__)
txt += "\n\n### Google mobility\n"
n_domains = 6
dummy_g_mob = pd.DataFrame(1.0, index=range(10), columns=range(n_domains))
dummy_g_priors = {"mob_weights": dist.Uniform(np.zeros(n_domains), np.ones(n_domains)), "mob_exp": None}
weighted_exp_mob_provider = WeightedExpMobilityProvider(dummy_g_mob, dummy_g_priors)
txt += get_func_blurb(process_raw_google_mobility)
txt += get_func_blurb(get_google_mobility)
txt += get_func_blurb(weighted_exp_mob_provider.__init__)
txt += get_func_blurb(weighted_exp_mob_provider.get_parameterised_mobility)
txt += "\n\n### Facebook mobility\n"
dummy_fb_mob_builder = FacebookMobilityBuilder()
txt += get_func_blurb(dummy_fb_mob_builder.build_mobility) + "\n\n"
txt += get_func_blurb(get_fb_visited_mobility)
txt += get_func_blurb(get_fb_singletile_mobility) + "\n\n"
dummy_fb_mob = pd.Series(1.0, index=range(10))
dummy_fb_priors = {"mob_exp": None}
single_series_mob_provider = SingleSeriesExpMobilityProvider(dummy_fb_mob, dummy_fb_priors)
txt += get_func_blurb(single_series_mob_provider.__init__)
txt += get_func_blurb(single_series_mob_provider.get_parameterised_mobility)
dummy_start = datetime(2020, 1, 1)
dummy_end = datetime(2021, 12, 31)
dummy_model = MultiStrainModel(1.0, dummy_start, dummy_end, [""], dummy_start, no_mob_provider, False)
txt += "\n\n## Variable process\n"
process_rationale = (
    "We included a non-mechanistic variable process "
    "within our model, which was intended to capture "
    "any variations in transmission that could not be "
    "attributed to the explicitly modelled processes "
    "(variant strain emergence, changes in mobility "
    "and the development of population immunity).\n\n"
)
txt += get_func_blurb(dummy_model.initialise_var_proc)
txt += get_func_blurb(_get_cos_curve_at_x)
txt += get_func_blurb(dummy_model.fit_process_curve)
txt += "\n\n## Renewal model\n"
model_rationale = (
    "We developed a discrete renewal model "
    "with daily innovations. The core aspect of our "
    "approach was the convolution of the preceding "
    "strain-specific incidence rate with "
    "the generation interval for subsequent infections, "
    "adjusted for past immunity. "
    "We then applied convolutions to this model "
    "to estimate model outputs that could be compared "
    "against our indicator targets introduced above.\n\n"
)
txt += model_rationale
txt += get_func_blurb(dummy_model.renew)
dummy_dist = GammaDens()
txt += get_func_blurb(dummy_dist.get_params) + "\n\n"
txt += "\n\n### Outputs\n"
txt += get_func_blurb(dummy_model.renewal_func)
txt += "\n\n## Calibration\n"
calibration_rationale = (
    "We used a non-gradient-based algorithm "
    "for model calibration, which efficiently "
    "explored our multidimensional parameter space. "
    "By exploring all time series in log-space "
    "with a common dispersion parameter, "
    "we were able to apply an algorithm that "
    "provided epidemiologically plausible results for all "
    "countries simulated without the need for "
    "extensive country-specific adaptations.\n\n"
)
txt += calibration_rationale
txt += "\n\n### Initialisation\n"
txt += get_func_blurb(custom_init)
txt += "\n\n### Parameter processing\n"
dummy_calib = StandardCalib(dummy_model, {}, {})
txt += get_func_blurb(dummy_calib.__init__)
txt += get_func_blurb(dummy_calib.sample_calib_params)
txt += get_func_blurb(get_standard_priors)
txt += "\n\n### Algorithm\n"
txt += get_func_blurb(run_calibration)

In [ ]:
display(Markdown(txt))

In [ ]:
loaded_priors = yml.safe_load(open(BASE_PATH / "data/config/priors.yml", "r"))

{{< pagebreak >}}

In [ ]:
#| label: tbl-props
#| tbl-cap: Parameters and supporting evidence to proportion priors set from summary statistics.
#| tbl-colwidths: [20, 10, 10, 60]
betasum_table = get_table_df_from_priors_dict(loaded_priors["beta_from_summary"])
Markdown(betasum_table.to_markdown())

In [ ]:
#| label: tbl-betaprops
#| tbl-cap: Parameters and supporting evidence to proportion priors set from beta distribution parameters.
#| tbl-colwidths: [20, 10, 10, 60]
beta_table = get_table_df_from_priors_dict(loaded_priors["beta"])
Markdown(beta_table.to_markdown())

In [ ]:
#| label: tbl-durs
#| tbl-cap: Parameters and supporting evidence to time period priors.
#| tbl-colwidths: [20, 10, 10, 60]
dur_table = get_table_df_from_priors_dict(loaded_priors["durations"])
Markdown(dur_table.to_markdown())

In [ ]:
txt = "# Parameter evidence\n"
txt += (
    "The evidence used to justify the selection of the "
    "prior distributions to the time period parameters "
    "is presented in @tbl-durs. "
)
display(Markdown(txt))

In [ ]:
txt = "# Selection of countries for inclusion\n"
either_mob_avail, summary, g_avail, fb_avail = get_mob_avail_countries()
txt += get_func_blurb(get_mob_avail_countries)
death_data, case_data = gather_who_data(either_mob_avail)
txt += get_func_blurb(gather_who_data) + "\n\n"
no_deaths, no_cases = find_absent_inds(death_data, case_data, summary)
txt += get_func_blurb(find_absent_inds)
neg_deaths, neg_cases = find_neg_inds(death_data, case_data, summary)
txt += get_func_blurb(find_neg_inds)
death_outliers, case_outliers = find_outliers(death_data, case_data, summary)
death_nans, case_nans, death_repeats, case_repeats = find_nans_repeats(death_data, case_data, summary)
txt += get_func_blurb(find_nans_repeats) + "\n\n"
txt += "The final results of the selection are presented in @tbl-inc ."
txt += "The final included countries are illustrated in @fig-inc."
display(Markdown(txt))

In [ ]:
# fig_text = "The final included countries are illustrated in @fig-inc."
# Markdown(fig_text)

In [ ]:
#| label: tbl-inc
#| tbl-cap: Reasons for country inclusion and exclusion.
summary.index = summary.index.map(lambda iso3: pycountry.countries.lookup(iso3).name)
display(Markdown(summary.to_markdown()))

In [ ]:
excluded = set(no_deaths + no_cases + neg_deaths + neg_cases + death_nans + case_nans + death_repeats + case_repeats + death_outliers + case_outliers)
included = [c for c in either_mob_avail if c not in excluded]
add_bool_row_to_table(summary, included, "Incl-uded")
included.append("AUS")

In [ ]:
#| label: fig-inc
#| fig-cap: Included countries and mobility availability. Countries coloured according to mobility availability. Neither source available, grey; Google mobility only available, green; Facebook mobility only available, blue; both sources available, purple. Red markers indicate included in analysis.
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1)
add_mob_avail_to_world(world, g_avail, fb_avail)
world["included"] = world["ISO_A3"].isin(included)
plot_inclusion(world);

{{< pagebreak >}}

# References